In [2]:
import pandas as pd
import numpy as np
from pyomo.environ import *
from pyomo.dae import *
import matplotlib.pyplot as plt

# 1. Configurações e Dados
file_name = 'dados2.xlsx'
df = pd.read_excel(file_name)

# Pressão total do sistema 
P_total = 47.5  # bar

# Lumps seguindo a lógica do artigo (ajuste os nomes conforme seu Excel)
lumps_n   = ['Hn', 'Dn', 'Qn', 'Nn']   # n-parafinas (C22+, C15-22, C10-14, C5-9)
lumps_iso = ['Hiso', 'Diso', 'Qiso', 'Niso'] # iso-parafinas
lumps_G   = ['G']                      # C1-4 (gases)
lumps     = lumps_n + lumps_iso + lumps_G

df[lumps] = df[lumps].div(df[lumps].sum(axis=1), axis=0)
t_exp = df['x'].values
dados_exp = {lump: df[lump].values for lump in lumps}
val_iniciais = {lump: dados_exp[lump][0] for lump in lumps}

def solve_estimation():
    m = ConcreteModel()

    # Conjuntos
    m.t = ContinuousSet(initialize=t_exp)
    m.lump = Set(initialize=lumps)

    # 2. Variáveis de Decisão: Constantes cinéticas diretas (k)
    # k1-k4: Isomerização | k5-k8: Craqueamento
    m.k = Var(RangeSet(1, 8), initialize=0.1, bounds=(1e-6, 1e4))

    # Variáveis de estado e pressões parciais
    m.C = Var(m.lump, m.t, within=NonNegativeReals)
    m.dCdt = DerivativeVar(m.C, wrt=m.t)

    # Função auxiliar para pressão parcial (pi = Ci * Ptotal) 
    def _p(m, lump, t):
        return m.C[lump, t] * P_total

    # 3. Balanços de Massa (Baseado na Fig. 3 do artigo) 
    # k1: nC22 <-> iC22 | k2: nC15-22 <-> iC15-22 | k3: nC10-14 <-> iC10-14 | k4: nC5-9 <-> iC5-9
    # k5: iC22 -> iC15 | k6: iC15 -> iC10 | k7: iC10 -> iC5 | k8: iC5 -> G
    
    def _balanco(m, l, t):
        # n-parafinas: apenas isomerização 
        if l == 'Nn':
            return m.dCdt[l, t] == - m.k[1] * _p(m, 'Nn', t)
        elif l == 'Qn':
            return m.dCdt[l, t] == - m.k[2] * _p(m, 'Qn', t)
        elif l == 'Dn':
            return m.dCdt[l, t] == - m.k[3] * _p(m, 'Dn', t)
        elif l == 'Hn':
            return m.dCdt[l, t] == - m.k[4] * _p(m, 'Hn', t)

        # iso-parafinas: isomerização + craqueamento em cadeia [cite: 113, 146, 149, 152]
        elif l == 'Niso':
            return m.dCdt[l, t] == m.k[1]*_p(m, 'Nn', t) - m.k[5]*_p(m, 'Niso', t)
        elif l == 'Qiso':
            return m.dCdt[l, t] == m.k[2]*_p(m, 'Qn', t) + m.k[5]*_p(m, 'Niso', t) - m.k[6]*_p(m, 'Qiso', t)
        elif l == 'Diso':
            return m.dCdt[l, t] == m.k[3]*_p(m, 'Dn', t) + m.k[6]*_p(m, 'Qiso', t) - m.k[7]*_p(m, 'Diso', t)
        elif l == 'Hiso':
            return m.dCdt[l, t] == m.k[4]*_p(m, 'Hn', t) + m.k[7]*_p(m, 'Diso', t) - m.k[8]*_p(m, 'Hiso', t)

        # Produto Final G [cite: 153]
        elif l == 'G':
            return m.dCdt[l, t] == m.k[8] * _p(m, 'Hiso', t)

    m.ode = Constraint(m.lump, m.t, rule=_balanco)

    # Condições iniciais
    for l in lumps:
        m.C[l, t_exp[0]].fix(val_iniciais[l])

    # Função objetivo: Mínimos Quadrados
    def _obj(m):
        return sum((m.C[l, t] - dados_exp[l][np.where(t_exp == t)[0][0]])**2 
                   for l in lumps for t in t_exp)
    m.obj = Objective(rule=_obj, sense=minimize)

    # Discretização e Solver
    TransformationFactory('dae.collocation').apply_to(m, nfe=20, ncp=3)
    solver = SolverFactory('ipopt', executable=r'C:\zebra\ipopt.exe')
    solver.solve(m, tee=True)
    return m

# 4. Execução do Modelo
model = solve_estimation()

# 5. Cálculo do RMSE por espécie
rmse_por_especie = {}
erros_quadraticos_totais = []

for l in lumps:
    y_exp = dados_exp[l]
    y_model = np.array([value(model.C[l, t]) for t in t_exp])
    
    # Diferença residual
    residuos = y_exp - y_model
    rmse_por_especie[l] = np.sqrt(np.mean(residuos**2))
    
    # Armazena todos os resíduos para o cálculo global real
    erros_quadraticos_totais.extend(residuos**2)

# RMSE Global baseado em todos os pontos de todas as espécies
rmse_global = np.sqrt(np.mean(erros_quadraticos_totais))

# 6. Plotagem Gráfica
t_plot = sorted(list(model.t))

# Configurações globais para apresentação
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 13  # Tamanho ideal para caber bem dentro do quadro
plt.rcParams['lines.linewidth'] = 3.5

# Definição de estilos (Cores adaptadas para daltônicos - Okabe-Ito)
cores = {
    'Hn': '#0072B2',    # Azul Forte
    'Dn': '#D55E00',    # Vermelho
    'Qn': '#009E73',    # Verde-azulado
    'Nn': '#CC79A7',    # Púrpura/Rosa
    'Hiso': '#56B4E9',  # Azul Claro
    'Diso': '#E69F00',  # Laranja
    'Qiso': '#88CCEE',  # Ciano (substituindo o cinza claro)
    'Niso': '#882255',  # Vinho (substituindo o cinza escuro)
    'G': '#332288'      # Roxo Forte (substituindo o preto)
}

markers = {
    'Nn': 'o', 'Qn': '^', 'Dn': 'D', 'Hn': 'd',
    'Niso': 's', 'Qiso': 'v', 'Diso': 'P', 'Hiso': 'x',
    'G': '*'
}

tamanho_slide = (10, 7) # Altura excelente para slides

# =============================================================================
# GRÁFICO 1: Apenas Consumo de n-Parafinas
# =============================================================================
plt.figure(figsize=tamanho_slide)

for l in lumps_n:
    c = cores[l]
    mk = markers[l]
    plt.scatter(t_exp, dados_exp[l], color=c, marker=mk, alpha=0.8, s=120, label=f'{l} exp.')
    plt.plot(t_plot, [value(model.C[l, t]) for t in t_plot], color=c, label=f'{l} mod.')

plt.title('Consumo de n-Parafinas', fontweight='bold', pad=15)
plt.xlabel('Tempo (1/WHSV)', labelpad=10)
plt.ylabel('Fração Molar', labelpad=10)
plt.grid(True, linestyle='--', alpha=0.4)

plt.ylim(0, 0.35) 

# Legenda DENTRO do gráfico, no topo e dividida em colunas
plt.legend(loc='upper right', ncol=2) 
plt.tight_layout()

# Salva a primeira figura em 1200 DPI
plt.savefig('consumo_n_parafinas.png', dpi=1200, bbox_inches='tight')
# Opcional: Descomente a linha abaixo se quiser visualizar antes de fechar a figura
# plt.show() 
plt.close() # Fecha a figura da memória para não sobrepor com a próxima

# =============================================================================
# GRÁFICO 2: Produção de iso-Parafinas + Gases
# =============================================================================
plt.figure(figsize=tamanho_slide)

lumps_iso_G = lumps_iso + lumps_G

for l in lumps_iso_G:
    c = cores[l]
    mk = markers[l]
    estilo_linha = '-' if l == 'G' else '--'
    
    plt.scatter(t_exp, dados_exp[l], color=c, marker=mk, alpha=0.8, s=120, label=f'{l} exp.')
    plt.plot(t_plot, [value(model.C[l, t]) for t in t_plot], 
             color=c, linestyle=estilo_linha, label=f'{l} mod.')

plt.title('Produção de iso-Parafinas e Gases', fontweight='bold', pad=15)
plt.xlabel('Tempo (1/WHSV)', labelpad=10)
plt.ylabel('Fração Molar', labelpad=10) # Corrigi o label se estiver usando massa ou vice-versa, apenas confira se é molar mesmo.
plt.grid(True, linestyle='--', alpha=0.4)

plt.ylim(0, 0.45) 

# Legenda DENTRO do gráfico, dividida em 3 colunas para ficar compacta e larga
plt.legend(loc='upper center', ncol=3)
plt.tight_layout()

# Salva a segunda figura em 1200 DPI
plt.savefig('producao_iso_parafinas_gases.png', dpi=1200, bbox_inches='tight')
# Opcional: plt.show()
plt.close()

# 7. Exibição das Constantes Otimizadas
print("\n" + "="*45)
print(f" RMSE Global do Ajuste: {rmse_global:.5f}")
print("="*45)
print(" CONSTANTES CINÉTICAS (k)")
print("-"*45)
for i in range(1, 9):
    print(f" k{i:02d}: {value(model.k[i]):.4e}")

Ipopt 3.12.13: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

This is Ipopt version 3.12.13, running with linear solver mumps.
NOTE: Other linear solvers might be more efficient (see Ipopt documentation).

Number of nonzeros in equality constraint Jacobian...:     5150
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:      570

Total number of variables............................:     1097
                     variables with only lower bounds:      540
                variables with lower and upper bounds:        8
                     variables with only upper bounds:        0
Tot